# 05 — LODES Panel Builder

Builds the `did_lodes_panel.csv` used by `R/09_did_lodes.R`.

## Steps
1. Load raw LODES WAC tract data from `data/raw/lehd_lodes/lodes_wac_tracts.parquet`
2. Load NFIP claim crosswalk from the FEMA NFIP claims file (same as used in notebook 04)
3. Assign treatment (tracts with >$200K NFIP total paid) and compute intensity
4. Create outcome variables: `ln_jobs`, `ln_hi_wage`, sector columns
5. Export `data/processed/panels/did_lodes_panel.csv`

## Design Caveat
LODES data starts in 2002 — **4 years after the October 1998 flood**. There is no
pre-treatment period. This analysis tests whether flood-exposed tracts show **persistent
workplace differences** from 2002 onward, not the initial causal impact.

Only 2 tracts have NFIP payouts > $200K, so the analysis is underpowered.
Interpret results as descriptive/suggestive, not causal.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from src.utils.file_io import load_parquet
from src.utils.logging_setup import get_logger

log = get_logger('05_lodes_panel')

DATA_DIR = Path('../data')
RAW_DIR  = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed' / 'panels'
PROC_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load LODES WAC Data

In [ ]:
lodes_path = RAW_DIR / 'lehd_lodes' / 'lodes_wac_tracts.parquet'

if not lodes_path.exists():
    raise FileNotFoundError(
        f'LODES data not found: {lodes_path}\n'
        'Run: python -m src.pipeline --task lehd_lodes'
    )

lodes = load_parquet(lodes_path)
lodes['tract'] = lodes['tract'].astype(str).str.zfill(11)
lodes['year'] = lodes['year'].astype(int)

print(f'LODES WAC tracts: {lodes.shape}')
print(f'Tracts: {lodes["tract"].nunique()}')
print(f'Years: {lodes["year"].min()}–{lodes["year"].max()}')
lodes.head()

## 2. Load NFIP Claim Crosswalk

In [ ]:
# Load NFIP claims crosswalked to census tracts.
# This was built in notebook 04 (HPI DiD panel).
# If not available, load raw NFIP and re-crosswalk here.

nfip_tract_path = PROC_DIR / 'nfip_tract_damage.csv'

if nfip_tract_path.exists():
    nfip = pd.read_csv(nfip_tract_path, dtype={'censusTract': str})
    nfip['tract'] = nfip['censusTract'].str.zfill(11)
    print(f'NFIP tract crosswalk: {nfip.shape}')
    print(nfip[['tract', 'nfip_total_paid']].head())
else:
    # Fall back: load raw NFIP and aggregate to tract
    print('nfip_tract_damage.csv not found — attempting to build from raw NFIP data...')
    nfip_raw_path = RAW_DIR / 'fema_nfip'
    raw_files = list(nfip_raw_path.glob('*.parquet'))
    if not raw_files:
        raise FileNotFoundError('No NFIP raw data found. Run: make acquire-fema_nfip')

    nfip_raw = load_parquet(raw_files[0])

    # Filter to DR-1257 (Oct 1998) claims in Comal + Hays + Kendall counties
    target_fips = ['48091', '48209', '48259']
    if 'countyCode' in nfip_raw.columns:
        nfip_raw['fips'] = '48' + nfip_raw['countyCode'].astype(str).str.zfill(3)
    nfip_county = nfip_raw[nfip_raw['fips'].isin(target_fips)].copy()

    # Aggregate to census tract
    if 'censusTract' in nfip_county.columns and 'totalPayments' in nfip_county.columns:
        nfip_county['totalPayments'] = pd.to_numeric(nfip_county['totalPayments'], errors='coerce')
        nfip = nfip_county.groupby('censusTract', as_index=False)['totalPayments'].sum()
        nfip.columns = ['censusTract', 'nfip_total_paid']
        nfip['tract'] = nfip['censusTract'].astype(str).str.zfill(11)
        nfip.to_csv(nfip_tract_path, index=False)
        print(f'Built NFIP tract crosswalk: {nfip.shape}')
    else:
        raise ValueError(f'Expected censusTract + totalPayments columns. Found: {list(nfip_county.columns)[:10]}')

## 3. Assign Treatment and Intensity

In [ ]:
NFIP_TREAT_THRESH = 200_000  # Same threshold as R/04 HPI tract analysis

# Tracts present in LODES
lodes_tracts = lodes[['tract']].drop_duplicates()
print(f'Unique LODES tracts: {len(lodes_tracts)}')

# Merge NFIP damage (tracts with no claims get nfip_total_paid = 0)
tract_damage = lodes_tracts.merge(
    nfip[['tract', 'nfip_total_paid']], on='tract', how='left'
)
tract_damage['nfip_total_paid'] = tract_damage['nfip_total_paid'].fillna(0)

# Treatment assignment
tract_damage['treated']   = (tract_damage['nfip_total_paid'] > NFIP_TREAT_THRESH).astype(int)
tract_damage['intensity'] = np.log1p(tract_damage['nfip_total_paid'])

print(f'\nTreatment summary (threshold = ${NFIP_TREAT_THRESH:,}):')
print(tract_damage['treated'].value_counts().rename({0: 'control', 1: 'treated'}))
print('\nTreated tracts:')
print(tract_damage[tract_damage['treated'] == 1][['tract', 'nfip_total_paid']].to_string())

## 4. Merge and Build Panel

In [ ]:
# Merge treatment assignment into LODES panel
panel = lodes.merge(tract_damage[['tract', 'treated', 'intensity', 'nfip_total_paid']],
                    on='tract', how='left')

# Drop tracts with no treatment assignment (shouldn't happen but guard)
panel = panel.dropna(subset=['treated'])
panel['treated'] = panel['treated'].astype(int)

# Numeric columns
job_cols = ['C000', 'CE01', 'CE02', 'CE03'] + [f'CNS{i:02d}' for i in range(1, 21)]
for col in job_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors='coerce')

print(f'Panel shape: {panel.shape}')
print(f'Tracts: {panel["tract"].nunique()} ({panel["treated"].eq(1).groupby(panel["tract"]).first().sum()} treated)')
print(f'Years: {panel["year"].min()}–{panel["year"].max()}')
panel.head()

## 5. Descriptive Checks

In [ ]:
import matplotlib.pyplot as plt

# Total jobs trajectory: treated vs. control tracts
traj = panel.groupby(['year', 'treated'])['C000'].mean().reset_index()
traj['group'] = traj['treated'].map({0: 'Control tracts', 1: 'Treated tracts (>$200K NFIP)'})

fig, ax = plt.subplots(figsize=(10, 4))
for grp, sub in traj.groupby('group'):
    ax.plot(sub['year'], sub['C000'], marker='o', markersize=4, label=grp)
ax.axvline(2010, linestyle=':', color='steelblue', label='Post-2010 cutoff')
ax.set_title('Mean Workplace Jobs: Treated vs. Control Tracts (LODES WAC)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Mean jobs per tract')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/results/figures/15_lodes_trajectories_descriptive.png', dpi=150)
plt.show()

print('Note: No pre-2002 data available (flood was 1998). Cannot assess parallel trends.')

## 6. Export Panel

In [ ]:
out_path = PROC_DIR / 'did_lodes_panel.csv'

# Keep relevant columns
keep_cols = ['tract', 'year', 'treated', 'intensity', 'nfip_total_paid',
             'C000', 'CE01', 'CE02', 'CE03'] + [f'CNS{i:02d}' for i in range(1, 21)]
keep_cols = [c for c in keep_cols if c in panel.columns]

panel[keep_cols].sort_values(['tract', 'year']).to_csv(out_path, index=False)

print(f'Exported: {out_path}')
print(f'Shape: {panel[keep_cols].shape}')
print(f'Tracts: {panel["tract"].nunique()}')
print(f'Years: {panel["year"].min()}–{panel["year"].max()}')
print(f'Treated: {panel[panel["treated"]==1]["tract"].nunique()}')
print(f'Control: {panel[panel["treated"]==0]["tract"].nunique()}')
print('\nNext step: Run R/09_did_lodes.R')